In [1]:
import argparse
import os
import sys
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

# Google Colab: mount Google Drive and work from the copy of the software/ folder there (it must contain models/),
# so the imports below work and checkpoints in result/ survive a runtime reset. Skipped when running locally.
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    colab_dir = '/content/drive/MyDrive/ece284fa26/software'   # change to where you uploaded the software/ folder
    os.chdir(colab_dir)
    sys.path.insert(0, colab_dir)

from models import *


global best_prec
if torch.cuda.is_available():
    device = torch.device('cuda')
    cudnn.benchmark = True       # fixed 32x32 input -> let cuDNN pick the fastest conv kernels
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)

device: cuda


In [2]:
print('=> Building model...')
    
batch_size = 128
model_name = "resnet20_quant"
model = resnet20_quant()

print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [20, 40]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        



=> Building model...
ResNet_Cifar(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): QuantConv2d(
        16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (weight_quant): weight_quantize_fn()
      )
      (conv2): QuantConv2d(
        16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (weight_quant): weight_quantize_fn()
      )
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): QuantConv2d(
        16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
        (weight_quant): weight_quantize_fn()

In [ ]:
# This cell won't be given, but students will complete the training

lr = 1e-2
weight_decay = 1e-4
epochs = 50
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

In [3]:
PATH = "result/resnet20_quant/model_best.pth.tar"
checkpoint = torch.load(PATH, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # move to device
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8536/10000 (85%)



## (A) 4-bit: x_int, w_int and psum recovery for the 2nd conv layer (model.layer1[0].conv1)

In [4]:
# Capture the input and output of the 2nd conv layer (model.layer1[0].conv1) with a forward hook
conv = model.layer1[0].conv1

class SaveInOut:
    def __init__(self):
        self.input = None
        self.output = None
    def __call__(self, module, module_in, module_out):
        self.input = module_in[0].detach().clone()
        self.output = module_out.detach().clone()

save = SaveInOut()
handle = conv.register_forward_hook(save)

images, labels = next(iter(testloader))
model.eval()
with torch.no_grad():
    model(images.to(device))
handle.remove()

x = save.input              # floating-point input (the layer quantizes it internally)
output_layer = save.output  # the layer's own output
print(x.shape, output_layer.shape)

torch.Size([128, 16, 32, 32]) torch.Size([128, 16, 32, 32])


In [5]:
w_bit = 4
weight_q = conv.weight_q.detach()               # quantized weight stored during the forward pass (normalized units)
w_alpha = conv.weight_quant.wgt_alpha.detach()  # weight clipping value learned during training
w_delta = w_alpha / (2 ** (w_bit - 1) - 1)      # signed weights: integer levels -7..7
weight_int = (weight_q / w_delta).round()

print('max distance to nearest integer before rounding:', (weight_q / w_delta - weight_int).abs().max().item())
print('w_int values:', weight_int.unique().int().tolist())
print(weight_int[0, 0])  # you should see clean integer numbers

max distance to nearest integer before rounding: 4.76837158203125e-07
w_int values: [-7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7]
tensor([[ 4., -2.,  2.],
        [-3., -4.,  4.],
        [ 1., -2.,  2.]], device='cuda:0')


In [6]:
x_bit = 4
x_alpha = conv.act_alpha.detach()          # activation clipping value learned during training
x_delta = x_alpha / (2 ** x_bit - 1)       # unsigned activations: integer levels 0..15

act_quant_fn = act_quantization(x_bit)     # same activation quantizer the layer uses
with torch.no_grad():
    x_q = act_quant_fn(x, x_alpha)         # quantized value of x

x_int = (x_q / x_delta).round()
print('max distance to nearest integer before rounding:', (x_q / x_delta - x_int).abs().max().item())
print('x_int values:', x_int.unique().int().tolist())
print(x_int[0, 0, :8, :8])  # you should see clean integer numbers

max distance to nearest integer before rounding: 9.5367431640625e-07
x_int values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
tensor([[1., 2., 2., 2., 2., 2., 2., 2.],
        [2., 3., 4., 4., 4., 4., 4., 4.],
        [2., 3., 4., 4., 4., 4., 4., 4.],
        [2., 3., 4., 5., 4., 4., 4., 4.],
        [3., 4., 4., 4., 4., 4., 4., 4.],
        [2., 4., 4., 4., 4., 4., 4., 4.],
        [2., 1., 1., 6., 6., 4., 4., 4.],
        [2., 1., 0., 2., 5., 4., 4., 4.]], device='cuda:0')


In [7]:
# Integer convolution with the same stride and padding as the layer, then rescale with the step sizes
conv_int = nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1, bias=False).to(device)
conv_int.weight = nn.Parameter(weight_int)

with torch.no_grad():
    output_int = conv_int(x_int)                        # integer partial sums
    output_recovered = output_int * x_delta * w_delta   # psum_recovered = conv(x_int, w_int) * delta_x * delta_w
print(output_recovered[0, 0, :4, :4])

tensor([[11.1518, -0.3913, -2.7390, -1.7608],
        [17.8038, -0.1956,  5.2824,  5.4781],
        [17.6081, -0.7826,  0.3913,  4.3042],
        [13.1083, -4.6955,  1.3695,  2.1521]], device='cuda:0')


In [8]:
# Required check: the recovered psum vs. the layer's own output captured by the forward hook (should be ~0)
difference = (output_layer - output_recovered).abs()
print('mean abs difference: {:.3e}'.format(difference.mean().item()))
print('max  abs difference: {:.3e}'.format(difference.max().item()))

mean abs difference: 1.374e-06
max  abs difference: 3.052e-05


In [9]:
# For reference (as in [HW_Code5]): compare with floating-point-input versions of this layer.
# These gaps measure quantization error rather than a calculation error, so there is no target value.
with torch.no_grad():
    w = conv.weight
    ref_wq = F.conv2d(x, weight_q, padding=1)                    # float x, quantized weight
    ref_fp = F.conv2d(x, (w - w.mean()) / w.std(), padding=1)    # float x, float (normalized) weight
for name, ref in [('float x, quantized w', ref_wq), ('float x, float w', ref_fp)]:
    print('vs. {:<20s}: mean abs difference = {:.4f}'.format(name, (ref - output_recovered).abs().mean().item()))

vs. float x, quantized w: mean abs difference = 1.1267
vs. float x, float w    : mean abs difference = 1.6784


## (B) 2-bit: 2-bit weights and activations, first conv layer quantized as well

In [10]:
from models.resnet_quant_2bit import resnet20_quant_2bit
from models.quant_layer_2bit import QuantConv2d as QuantConv2d_2bit

model_name_2bit = "resnet20_quant_2bit"
model_2bit = resnet20_quant_2bit()

print(model_2bit.conv1)  # the first conv layer is now a 2-bit QuantConv2d
qconvs = [m for m in model_2bit.modules() if isinstance(m, QuantConv2d_2bit)]
print('{} QuantConv2d layers, bit widths: {}'.format(len(qconvs), sorted({m.bit for m in qconvs})))

QuantConv2d(
  3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, bit=2
  (weight_quant): weight_quantize_fn()
)
21 QuantConv2d layers, bit widths: [2]


In [ ]:
for name, m in model_2bit.named_modules():
  if isinstance(m, QuantConv2d_2bit):
      m.weight_quant.wgt_alpha.data.fill_(1.5)
      m.act_alpha.data.fill_(2.0 if name == 'conv1' else 3.0)

lr = 0.1
weight_decay = 1e-4
epochs = 50
best_prec_2bit = 0

model_2bit.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer_2bit = torch.optim.SGD(model_2bit.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

fdir_2bit = 'result/' + model_name_2bit
if not os.path.exists(fdir_2bit):
    os.makedirs(fdir_2bit)


for epoch in range(0, epochs):
    adjust_learning_rate(optimizer_2bit, epoch)

    train(trainloader, model_2bit, criterion, optimizer_2bit, epoch)

    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model_2bit, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec_2bit
    best_prec_2bit = max(prec, best_prec_2bit)
    print('best acc: {:1f}'.format(best_prec_2bit))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model_2bit.state_dict(),
        'best_prec': best_prec_2bit,
        'optimizer': optimizer_2bit.state_dict(),
    }, is_best, fdir_2bit)

In [11]:
PATH = "result/resnet20_quant_2bit/model_best.pth.tar"
checkpoint = torch.load(PATH, map_location=device)
model_2bit.load_state_dict(checkpoint['state_dict'])
model_2bit.to(device)
model_2bit.eval()

correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # move to device
        output = model_2bit(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8046/10000 (80%)



In [12]:
# Confirm the bit widths: 2-bit weights should be integers in -1..1 and 2-bit activations in 0..3
layers_2bit = {'conv1': model_2bit.conv1, 'layer1[0].conv1': model_2bit.layer1[0].conv1}
inputs_2bit = {}

def save_input(name):
    def hook(module, module_in):
        inputs_2bit[name] = module_in[0].detach().clone()
    return hook

handles = [m.register_forward_pre_hook(save_input(n)) for n, m in layers_2bit.items()]
images, labels = next(iter(testloader))
model_2bit.eval()
with torch.no_grad():
    model_2bit(images.to(device))
for h in handles:
    h.remove()

with torch.no_grad():
    for name, m in layers_2bit.items():
        w_delta_2bit = m.weight_quant.wgt_alpha / (2 ** (m.bit - 1) - 1)
        x_delta_2bit = m.act_alpha / (2 ** m.bit - 1)
        w_int_2bit = (m.weight_q / w_delta_2bit).round()
        x_int_2bit = (act_quantization(m.bit)(inputs_2bit[name], m.act_alpha) / x_delta_2bit).round()
        print('{:<16s} w_int: {}   x_int: {}'.format(name, w_int_2bit.unique().int().tolist(),
                                                     x_int_2bit.unique().int().tolist()))

conv1            w_int: [-1, 0, 1]   x_int: [-3, -2, -1, 0, 1, 2, 3]
layer1[0].conv1  w_int: [-1, 0, 1]   x_int: [0, 1, 2, 3]


In [13]:
model_2bit.conv1

QuantConv2d(
  3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, bit=2
  (weight_quant): weight_quantize_fn()
)